# 📥 Kaggle Dataset Setup
## Run this ONCE to download the dataset to Google Drive

---
**You only need to run this notebook once.**
After the dataset is in your Drive, it stays there permanently.
All other notebooks just read from Drive — no re-downloading needed.

### Before running:
1. Go to kaggle.com → Profile → Settings → API → Create New Token
2. A file called `kaggle.json` downloads to your Mac
3. Have that file ready — Cell 3 will ask you to upload it

### Runtime setting:
Runtime → Change runtime type → **CPU** (no GPU needed here)

## CELL 1 — Mount Google Drive
Connect Colab to your Google Drive so the dataset saves permanently.
A popup will appear asking you to authorise — click Allow.

In [ ]:
# Mount Google Drive to Colab
# Everything saved to /content/drive/MyDrive/ persists between sessions
from google.colab import drive
drive.mount('/content/drive')

# Define your project root folder in Drive
# Change this if you named your folder differently
PROJECT = '/content/drive/MyDrive/knee_oa_kl_grading'

print(f'Project folder: {PROJECT}')
print('Drive mounted successfully')

## CELL 2 — Create the Folder Structure in Drive
This creates all the folders your project needs.
`exist_ok=True` means it won't crash if the folders already exist.

In [ ]:
import os

# List of all folders we need for the project
# We create them all now so nothing breaks later
folders = [
    f'{PROJECT}/data/raw',           # original Kaggle images go here
    f'{PROJECT}/data/processed',     # CLAHE-processed images (Experiment 1)
    f'{PROJECT}/outputs/checkpoints', # saved model weights after training
    f'{PROJECT}/outputs/figures',    # all plots and figures for the report
    f'{PROJECT}/outputs/results',    # CSV files with metrics from each experiment
    f'{PROJECT}/src',                # Python source code files
    f'{PROJECT}/experiments',        # config files for each experiment
    f'{PROJECT}/notebooks',          # Jupyter notebooks
    f'{PROJECT}/report',             # Quarto report files
]

# os.makedirs creates a folder and all parent folders needed
# exist_ok=True means: if folder already exists, don't raise an error
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print('Folder structure created in Google Drive:')
for folder in folders:
    # Check each folder was created successfully
    status = '✓' if os.path.exists(folder) else '✗ FAILED'
    # Show only the part after the project root to keep output readable
    short_name = folder.replace(PROJECT, '')
    print(f'  {status}  {short_name}')

## CELL 3 — Upload Your Kaggle API Key
This uploads `kaggle.json` from your Mac to Colab.
Colab will show a file picker — select your kaggle.json file.

**Your kaggle.json contains your private API key.**
It is only uploaded to your Colab session — it is not shared with anyone.

In [ ]:
# This imports the files module which lets you upload from your Mac
from google.colab import files

print('A file picker will appear below.')
print('Select your kaggle.json file from your Mac.')
print()

# files.upload() opens a file picker dialog
# It returns a dictionary of {filename: file_contents}
uploaded = files.upload()

# Check the right file was uploaded
if 'kaggle.json' in uploaded:
    print('kaggle.json uploaded successfully')
else:
    print('WARNING: file uploaded was not named kaggle.json')
    print('Make sure you selected the correct file')

## CELL 4 — Install Kaggle and Configure API Key
Kaggle is not pre-installed in Colab so we install it with pip.
Then we move your API key to the location Kaggle expects to find it.

In [ ]:
# Install the Kaggle Python package
# The -q flag means 'quiet' — it suppresses the installation output
# The ! at the start means 'run this as a terminal command, not Python'
!pip install kaggle -q

# Create the hidden .kaggle folder in the home directory
# This is where Kaggle looks for your API key by default
# The -p flag means 'create parent folders if needed'
!mkdir -p ~/.kaggle

# Copy your uploaded kaggle.json to the .kaggle folder
# kaggle.json was uploaded to /content/ (the default Colab working directory)
!cp /content/kaggle.json ~/.kaggle/

# Set the file permissions to read-only for the owner only
# chmod 600 means: owner can read+write, no access for anyone else
# This is required by the Kaggle API for security
!chmod 600 ~/.kaggle/kaggle.json

print('Kaggle API configured successfully')

# Verify it works by checking the version
!kaggle --version

## CELL 5 — Download the Dataset
This downloads the Kaggle Knee OA dataset directly to your Google Drive.

**Dataset:** Knee Osteoarthritis Dataset with Severity Grading
**Licence:** CC BY 4.0 (free to use for research)
**Size:** approximately 1.4 GB
**Time:** 3–5 minutes depending on your connection

The `--unzip` flag automatically extracts the zip file after downloading.

In [ ]:
# Define where we want to save the dataset
# We save directly to Drive so it persists permanently
DOWNLOAD_PATH = f'{PROJECT}/data/raw'

print(f'Downloading dataset to: {DOWNLOAD_PATH}')
print('This will take 3-5 minutes...')
print()

# Download the dataset using the Kaggle CLI
# kaggle datasets download: the download command
# -d shashwatwork/knee-osteoarthritis-dataset-with-severity: the dataset identifier
#    (this is the part of the Kaggle URL after kaggle.com/datasets/)
# -p: path to save to
# --unzip: automatically extract the zip file after downloading
!kaggle datasets download \
    -d shashwatwork/knee-osteoarthritis-dataset-with-severity \
    -p {DOWNLOAD_PATH} \
    --unzip

print()
print('Download complete')

## CELL 6 — Verify the Download
Check the folder structure looks right and count the images.
If something went wrong with the download, this cell will show you.

In [ ]:
import os
from pathlib import Path

# Define expected structure
# The Kaggle dataset should have train/val/test splits
# each containing subfolders 0, 1, 2, 3, 4 for KL grades
DATA_DIR = Path(f'{PROJECT}/data/raw')

print('Verifying dataset structure...')
print()

total_images = 0

# Check each split
for split in ['train', 'val', 'test']:
    split_path = DATA_DIR / split

    # Check the split folder exists
    if not split_path.exists():
        print(f'  ✗ {split}/ folder NOT FOUND — check download')
        continue

    split_total = 0
    grade_counts = []

    # Check each grade subfolder
    for grade in ['0', '1', '2', '3', '4']:
        grade_path = split_path / grade

        if grade_path.exists():
            # Count only image files (jpg or png)
            # A list comprehension creates a list in one line
            # f.endswith checks if the filename ends with that extension
            n = len([
                f for f in os.listdir(grade_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])
            grade_counts.append(f'KL{grade}={n}')
            split_total += n
        else:
            grade_counts.append(f'KL{grade}=MISSING')

    total_images += split_total
    print(f'  ✓ {split:6s}: {split_total:5d} images  |  {", ".join(grade_counts)}')

print()
print(f'Total images found: {total_images}')

# Validate against expected counts from the dataset documentation
EXPECTED_TOTAL = 9786
if total_images == EXPECTED_TOTAL:
    print(f'✓ Count matches expected {EXPECTED_TOTAL} images')
elif total_images > 0:
    print(f'⚠ Found {total_images} images, expected ~{EXPECTED_TOTAL}')
    print('  Download may be incomplete — try running Cell 5 again')
else:
    print('✗ No images found — check the folder structure above')
    print('  The zip may have extracted to a subfolder')
    print(f'  Try: !ls {DOWNLOAD_PATH} to see what was downloaded')

## CELL 7 — Handle Nested Folder (if needed)
Sometimes Kaggle extracts the zip into a subfolder with the dataset name.
Run this cell ONLY if Cell 6 showed zero images or missing folders.

In [ ]:
# This cell fixes the case where Kaggle extracted to a subfolder
# For example: data/raw/knee-osteoarthritis-dataset-with-severity/train/
# We want:     data/raw/train/

# First, see what actually got downloaded
print('Contents of data/raw folder:')
!ls -la {PROJECT}/data/raw/

print()
print('If you see a subfolder above (not train/val/test directly),')
print('uncomment and run the lines below, replacing SUBFOLDER_NAME')
print('with whatever subfolder name you see.')

# UNCOMMENT THESE LINES IF NEEDED:
# Replace 'SUBFOLDER_NAME' with the actual folder name shown above

# SUBFOLDER_NAME = 'knee-osteoarthritis-dataset-with-severity'
# !mv {PROJECT}/data/raw/{SUBFOLDER_NAME}/* {PROJECT}/data/raw/
# !rmdir {PROJECT}/data/raw/{SUBFOLDER_NAME}
# print('Files moved successfully — run Cell 6 again to verify')

## ✅ Setup Complete

If Cell 6 showed ~9,786 total images with all grades present, your dataset is ready.

**You never need to run this notebook again.**
The dataset is now permanently in your Google Drive.

---
### Next step
Open **`01_data_exploration.ipynb`** and run the full EDA.

### Your dataset citation for the report:
```
Shashwat (2021). Knee Osteoarthritis Dataset with Severity Grading.
Kaggle. https://www.kaggle.com/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity
Licence: CC BY 4.0. Source: Osteoarthritis Initiative (OAI).
```